In [1]:
"""
Script 1 — Native vs. Alanine-Scan Mutants
===========================================
Dataset: predictions_3ab.csv (multi-interface: A,C and B,C per sample)
 
Steps:
  1.  Load & filter interfaces (keep A,C and B,C, drop A,B)
  2.  Parse labels + base extraction  (binding column as ground truth)
  3.  Parse array-valued Chai columns
  4.  NaN audit + class imbalance report
  5.  Select score columns + auto-direction via ROC-AUC
  6.  Raw vs non-raw comparison
  7.  EDA: per-model KDE distributions + correlation heatmap
  8.  ΔΔScore (vectorized, per interface, with sanity-check flag)
  9.  Benchmarking: ROC-AUC, PR-AUC, F1, Precision@Opt — per interface
  10. Visualisations (grouped per model):
        - Strip + box separability plots
        - Quadrant plots (TP/FP/TN/FN) with sample labels
        - Top-metrics bar chart (PR-AUC + ROC-AUC)
        - PR curves for top-5 metrics per interface
        - Confusion matrix for best metric
        - ΔΔScore heatmaps (affected vs control interface)
"""
 
import re, ast, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, f1_score,
    precision_recall_curve, confusion_matrix,
    auc as auc_fn,
)
 
warnings.filterwarnings("ignore", category=UserWarning)
sns.set_theme(style="whitegrid")
 
# ══════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════
 
INPUT_CSV       = "/home/bri/pymol/prediction/data/predictions_3ab.csv"
OUTPUT_DIR      = Path("output_native_vs_mutants")
OUTPUT_DIR.mkdir(exist_ok=True)
 
KEEP_INTERFACES = {"A,C", "B,C"}   # A,B = antibody internal → always dropped
MIN_SAMPLES     = 10               # min rows needed to compute AUC
 
MUTATION_PATTERNS = [              # order matters: most specific first
    re.compile(r"_all_[A-Z]+$"),           # _all_A
    re.compile(r"_[A-Z]_[A-Z]\d+[A-Z]$"), # _A_H52A
]
MODEL_PREFIXES = ["af3", "cf", "esmfold", "chai", "boltz", "pred"]
 
BINDER_COLOR    = "#2196F3"
NONBINDER_COLOR = "#FF5722"
PALETTE_BOOL    = {True: BINDER_COLOR, False: NONBINDER_COLOR}
STRIP_PALETTE   = {"binder": BINDER_COLOR, "non-binder": NONBINDER_COLOR}
 
# ══════════════════════════════════════════════════════════════════
# STEP 1 — Load & Filter Interfaces
# ══════════════════════════════════════════════════════════════════
 
print("=" * 65)
print("STEP 1 — Load & Filter Interfaces")
print("=" * 65)
 
df = pd.read_csv(INPUT_CSV)
print(f"Raw shape: {df.shape}")
 
df = df[df["interface"].apply(
    lambda x: str(x).strip() in KEEP_INTERFACES)].copy()
interfaces = sorted(df["interface"].unique().tolist())
print(f"After interface filter {KEEP_INTERFACES}: {df.shape}")
print(f"Rows per interface:\n{df['interface'].value_counts().to_string()}")
 
# ══════════════════════════════════════════════════════════════════
# STEP 2 — Parse Labels & Base Extraction
# ══════════════════════════════════════════════════════════════════
 
print("\n" + "=" * 65)
print("STEP 2 — Parse Labels & Base Extraction")
print("=" * 65)
 
# Ground truth comes from 'binding' column — handles all mutant types:
#   WT binders:           5ig7_abc_cut          -> True
#   Single-point mutants: 5ig7_abc_cut_A_H52A   -> False
#   All-interface mutant: 5ig7_abc_cut_all_A    -> False
 
df["binding"] = df["binding"].map(
    {True: True, False: False, "True": True, "False": False,
     1: True, 0: False, "1": True, "0": False})
 
def extract_base(sample):
    for pat in MUTATION_PATTERNS:
        m = pat.search(str(sample))
        if m:
            return str(sample)[:m.start()]
    return str(sample)
 
def extract_mutation(sample):
    for pat in MUTATION_PATTERNS:
        m = pat.search(str(sample))
        if m:
            return m.group().lstrip("_")
    return None
 
def extract_mut_chain(sample):
    m = re.search(r"_([A-Z])_[A-Z]\d+[A-Z]$", str(sample))
    return m.group(1) if m else None
 
df["group"]     = df["binding"].map({True: "binder", False: "ala_scan"})
df["base"]      = df["sample"].apply(extract_base)
df["mutation"]  = df["sample"].apply(extract_mutation)
df["mut_chain"] = df["sample"].apply(extract_mut_chain)
 
print(df["group"].value_counts().to_string())
print(f"\nUnique base structures: {df['base'].nunique()}")
print(f"Unique mutations:       {df['mutation'].dropna().nunique()}")
print(f"\nParsing preview (first 8 rows):")
print(df[["sample", "interface", "group", "base",
          "mut_chain", "mutation"]].head(8).to_string(index=False))
 
# ══════════════════════════════════════════════════════════════════
# STEP 3 — Parse Array-Valued Chai Columns
# ══════════════════════════════════════════════════════════════════
 
print("\n" + "=" * 65)
print("STEP 3 — Parse Array-Valued Columns")
print("=" * 65)
 
def parse_array_column(series):
    """
    Converts '[[0.94, 0.88]]' strings to a scalar.
    Takes the MAX value — most optimistic score across chain pairs.
    """
    out = []
    for val in series:
        if pd.isna(val):
            out.append(np.nan); continue
        try:
            lst  = ast.literal_eval(str(val))
            flat = list(pd.core.common.flatten(lst))
            nums = [float(x) for x in flat if x is not None]
            out.append(max(nums) if nums else np.nan)
        except (ValueError, SyntaxError):
            out.append(np.nan)
    return out
 
array_cols = [
    c for c in df.columns
    if df[c].dtype == "object"
    and df[c].astype(str).str.contains(r"\[\[", na=False).any()
]
if array_cols:
    print(f"Found {len(array_cols)} array-valued columns — extracting max:")
    for col in array_cols:
        print(f"  {col}")
        df[col] = parse_array_column(df[col])
else:
    print("No array-valued columns found.")
 
# ══════════════════════════════════════════════════════════════════
# STEP 4 — NaN Audit & Class Imbalance Report
# ══════════════════════════════════════════════════════════════════
 
print("\n" + "=" * 65)
print("STEP 4 — NaN Audit & Class Imbalance Report")
print("=" * 65)
 
report_lines = []
for iface in interfaces + ["combined"]:
    sub = df if iface == "combined" else df[df["interface"] == iface]
    n_pos = sub["binding"].sum()
    n_neg = (~sub["binding"]).sum()
    total = len(sub)
    ratio = n_pos / n_neg if n_neg > 0 else float("inf")
    line  = (f"[{iface}]  Binders: {n_pos} ({n_pos/total*100:.1f}%)  "
             f"Non-binders: {n_neg} ({n_neg/total*100:.1f}%)  "
             f"Ratio: {ratio:.3f}")
    print(line)
    report_lines.append(line)
    if ratio < 0.3:
        advice = "  → HIGHLY IMBALANCED: use PR-AUC & F1, not ROC-AUC alone"
    elif ratio < 0.5:
        advice = "  → MODERATELY IMBALANCED: prefer PR-AUC"
    else:
        advice = "  → Relatively balanced"
    print(advice); report_lines.append(advice)
 
nan_counts = df.isnull().sum()
nan_summary = nan_counts[nan_counts > 0].sort_values(ascending=False)
print(f"\nColumns with NaN ({len(nan_summary)} total):")
print(nan_summary.to_string())
report_lines += ["", "NaN counts:"] + [f"  {k}: {v}"
                                        for k, v in nan_summary.items()]
 
with open(OUTPUT_DIR / "01_nan_imbalance_report.txt", "w") as f:
    f.write("\n".join(report_lines))
print("Saved → 01_nan_imbalance_report.txt")
 
# ══════════════════════════════════════════════════════════════════
# STEP 5 — Select Score Columns + Auto-Direction via ROC-AUC
# ══════════════════════════════════════════════════════════════════
 
print("\n" + "=" * 65)
print("STEP 5 — Select Score Columns + Auto-Direction")
print("=" * 65)
 
EXCLUDE = {
    "sample", "binding", "interface", "base", "group",
    "mut_chain", "mutation",
    # rank/ordinal — not continuous
    "af3_rank", "af3_rank0_masif", "af3_ranking_score",
    "cf_rank",  "cf_rank0_masif",
    # list-valued before parsing
    "chai_unconstrained_per_chain_ptm",
    "chai_unconstrained_per_chain_pair_iptm",
    "chai_constrained_per_chain_ptm",
    "chai_constrained_per_chain_pair_iptm",
}
path_cols = {c for c in df.columns if "path" in c.lower()}
EXCLUDE.update(path_cols)
 
score_cols = [
    c for c in df.columns
    if c not in EXCLUDE
    and pd.api.types.is_numeric_dtype(df[c])
    and df[c].nunique() > 1
]
 
model_groups = {p: [c for c in score_cols if c.startswith(p)]
                for p in MODEL_PREFIXES}
other = [c for c in score_cols
         if not any(c.startswith(p) for p in MODEL_PREFIXES)]
if other:
    model_groups["other"] = other
 
print(f"Score columns retained: {len(score_cols)}")
for m, cols in model_groups.items():
    if cols: print(f"  {m}: {len(cols)}")
 
# Auto-direction: fit on combined data (all interfaces)
# If ROC-AUC >= 0.5 -> higher is better (keep); else flip sign.
# NOTE: for ML, fit direction on training data only.
print("\nAuto-direction via ROC-AUC (combined interfaces)...")
y_all      = df["binding"].astype(int)
DIRECTIONS = {}
dir_rows   = []
 
for col in score_cols:
    s    = pd.to_numeric(df[col], errors="coerce")
    mask = s.notna() & y_all.notna()
    if mask.sum() < MIN_SAMPLES or y_all[mask].nunique() < 2:
        DIRECTIONS[col] = 1
        dir_rows.append({"col": col, "raw_roc": np.nan,
                         "direction": 1, "note": "insufficient_data"})
        continue
    raw_roc   = roc_auc_score(y_all[mask], s[mask])
    direction = 1 if raw_roc >= 0.5 else -1
    DIRECTIONS[col] = direction
    dir_rows.append({"col": col, "raw_roc": round(raw_roc, 4),
                     "direction": direction,
                     "note": "flipped" if direction == -1 else "kept"})
 
dir_df = pd.DataFrame(dir_rows)
dir_df.to_csv(OUTPUT_DIR / "02_auto_directions.csv", index=False)
print(dir_df[["col", "raw_roc", "direction", "note"]].to_string(index=False))
print(f"\nFlipped: {(dir_df['direction'] == -1).sum()} / {len(dir_df)}")
print("Saved → 02_auto_directions.csv")
 
# Apply directions to create aligned dataframe
df_aligned = df.copy()
for col in score_cols:
    if DIRECTIONS[col] == -1:
        df_aligned[col] = df_aligned[col] * -1
 
df_aligned.to_csv(OUTPUT_DIR / "scores_clean.csv", index=False)
print("Saved direction-aligned scores → scores_clean.csv")
 
# ══════════════════════════════════════════════════════════════════
# STEP 6 — Raw vs Non-Raw Comparison
# ══════════════════════════════════════════════════════════════════
 
print("\n" + "=" * 65)
print("STEP 6 — Raw vs Non-Raw Agreement (report only, no dropping)")
print("=" * 65)
 
RAW_ONLY = ("af3_notpl_raw_", "cf_custom_raw_", "cf_notpl_raw_")
 
def find_processed(raw_col, all_cols):
    if any(raw_col.startswith(p) for p in RAW_ONLY):
        return None
    m = re.match(r"^(af3|cf)_raw_(.+)$", raw_col)
    if m:
        c = f"{m.group(1)}_{m.group(2)}"
        return c if c in all_cols else None
    return None
 
all_cols = set(df.columns)
pairs    = [(find_processed(rc, all_cols), rc)
            for rc in df.columns if "_raw_" in rc
            and find_processed(rc, all_cols)]
 
print(f"Matched pairs: {len(pairs)}")
pair_rows = []
for col, raw_col in pairs:
    a    = pd.to_numeric(df[col],     errors="coerce")
    b    = pd.to_numeric(df[raw_col], errors="coerce")
    mask = a.notna() & b.notna()
    if mask.sum() < 2:
        pair_rows.append({"col": col, "raw_col": raw_col,
                          "corr": np.nan, "max_diff": np.nan,
                          "status": "insufficient_data"})
        continue
    corr     = a[mask].corr(b[mask])
    max_diff = (a[mask] - b[mask]).abs().max()
    if   corr > 0.9999 and max_diff < 1e-8: status = "IDENTICAL"
    elif corr > 0.99:                        status = "near_identical"
    else:                                    status = "DIFFERENT"
    pair_rows.append({"col": col, "raw_col": raw_col,
                      "corr": round(corr, 6),
                      "max_diff": round(max_diff, 8),
                      "status": status})
 
if pair_rows:
    pair_df = pd.DataFrame(pair_rows)
    print(pair_df[["col", "corr", "max_diff", "status"]].to_string(index=False))
    pair_df.to_csv(OUTPUT_DIR / "03_raw_vs_nonraw.csv", index=False)
    print("Saved → 03_raw_vs_nonraw.csv")
 
    # Scatter plots for DIFFERENT pairs
    diff_pairs = pair_df[pair_df["status"] == "DIFFERENT"]
    if len(diff_pairs):
        nc  = min(3, len(diff_pairs))
        nr  = int(np.ceil(len(diff_pairs) / nc))
        fig, axes = plt.subplots(nr, nc, figsize=(nc*4, nr*4), squeeze=False)
        axes = axes.flatten()
        for i, (_, row) in enumerate(diff_pairs.iterrows()):
            ax = axes[i]
            a  = pd.to_numeric(df[row["col"]],     errors="coerce")
            b  = pd.to_numeric(df[row["raw_col"]], errors="coerce")
            for val, color in [(True, BINDER_COLOR), (False, NONBINDER_COLOR)]:
                mask = df["binding"] == val
                ax.scatter(a[mask], b[mask], c=color,
                           label=("binder" if val else "non-binder"),
                           alpha=0.6, s=20, edgecolors="none")
            lims = [min(a.min(), b.min()), max(a.max(), b.max())]
            ax.plot(lims, lims, "k--", lw=0.8, alpha=0.5)
            ax.set_xlabel(f"processed: {row['col']}", fontsize=8)
            ax.set_ylabel(f"raw: {row['raw_col']}", fontsize=8)
            ax.set_title(f"r = {row['corr']:.4f}", fontsize=9)
            ax.legend(fontsize=7)
        for j in range(i + 1, len(axes)):
            axes[j].set_visible(False)
        fig.suptitle("Processed vs Raw — DIFFERENT pairs", fontsize=11)
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / "03b_raw_vs_processed_scatter.png",
                    dpi=150, bbox_inches="tight")
        plt.close()
        print("Saved → 03b_raw_vs_processed_scatter.png")
else:
    print("No matched raw/non-raw pairs found.")
 
# ══════════════════════════════════════════════════════════════════
# STEP 7 — EDA: Distributions + Correlation Heatmap
# ══════════════════════════════════════════════════════════════════
 
print("\n" + "=" * 65)
print("STEP 7 — EDA Plots")
print("=" * 65)
 
# 7a — Per-model KDE distributions (direction-aligned)
for model, cols in model_groups.items():
    valid = [c for c in cols if df_aligned[c].notna().any()]
    if not valid: continue
    nc   = 3
    nr   = (len(valid) + nc - 1) // nc
    fig, axes = plt.subplots(nr, nc, figsize=(15, 4 * nr))
    axes = np.atleast_1d(axes).flatten()
    for i, col in enumerate(valid):
        ax = axes[i]
        for val, color in [(True, BINDER_COLOR), (False, NONBINDER_COLOR)]:
            vals = df_aligned.loc[df_aligned["binding"] == val, col].dropna()
            if len(vals) > 1:
                vals.plot.kde(ax=ax, color=color,
                              label="binder" if val else "non-binder",
                              linewidth=2, alpha=0.25)
        ax.set_title(col, fontsize=9, fontweight="bold")
        ax.set_xlabel("")
        ax.legend(fontsize=7)
        ax.tick_params(labelsize=7)
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])
    fig.suptitle(f"{model.upper()} — Score Distributions (direction-aligned)",
                 fontsize=14, y=1.01)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"04_{model}_distributions.png",
                dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Saved → 04_{model}_distributions.png")
 
# 7b — Correlation heatmap (binders only)
key_metrics = [c for c in score_cols if any(k in c for k in
    ["iptm", "plddt", "ipae", "interface_dG", "ipsae",
     "confidence_score", "pde"])][:24]
 
binder_data = df_aligned.loc[df_aligned["binding"] == True,
                              key_metrics].dropna(axis=1, how="all")
if binder_data.shape[0] > 1 and binder_data.shape[1] > 1:
    corr = binder_data.corr()
    sz   = max(8, len(corr) * 0.5)
    fig, ax = plt.subplots(figsize=(sz, sz * 0.85))
    sns.heatmap(corr, ax=ax, cmap="RdBu_r", center=0, vmin=-1, vmax=1,
                annot=len(corr) <= 16, fmt=".2f", annot_kws={"size": 7},
                xticklabels=corr.columns, yticklabels=corr.index)
    ax.set_title("Metric Correlation — Binders only (direction-aligned)",
                 fontsize=12)
    plt.xticks(rotation=45, ha="right", fontsize=8)
    plt.yticks(fontsize=8)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "04z_correlation_heatmap.png",
                dpi=150, bbox_inches="tight")
    plt.close()
    print("Saved → 04z_correlation_heatmap.png")
 
# ══════════════════════════════════════════════════════════════════
# STEP 8 — ΔΔScore per Interface (REPAIRED)
# ══════════════════════════════════════════════════════════════════

print("\n" + "=" * 65)
print("STEP 8 — ΔΔScore per Interface")
print("=" * 65)

# 1. Create a clean WT reference (averaging seeds/models if duplicates exist)
wt_ref = (df_aligned[df_aligned["group"] == "binder"]
          .groupby(["base", "interface"])[score_cols]
          .mean()
          .reset_index())

# 2. Get the mutants
mutants = df_aligned[df_aligned["group"] == "ala_scan"].copy()

if wt_ref.empty or mutants.empty:
    print("Missing WT or Mutant data. Skipping ΔΔScore.")
    df_delta = None
else:
    # 3. Merge WT values into the Mutant dataframe
    # This aligns every mutant with its corresponding WT base/interface
    df_delta = mutants.merge(
        wt_ref, 
        on=["base", "interface"], 
        suffixes=("", "_wt"), 
        how="inner"
    )

    # 4. Calculate Delta: (Mutant Score - WT Score)
    # Since scores are direction-aligned, negative = mutation made it worse.
    for col in score_cols:
        df_delta[col] = df_delta[col] - df_delta[f"{col}_wt"]

    # 5. Define Affected Interface logic
    def is_expected_affected(row):
        if pd.isna(row["mut_chain"]):
            return True  # e.g., _all_A mutant affects all interfaces involving A
        return str(row["mut_chain"]) in str(row["interface"])

    df_delta["expected_affected"] = df_delta.apply(is_expected_affected, axis=1)

    # 6. Cleanup: drop the temporary _wt columns
    wt_cols = [f"{c}_wt" for c in score_cols]
    df_delta.drop(columns=wt_cols, inplace=True)

    df_delta.to_csv(OUTPUT_DIR / "scores_delta.csv", index=False)
    print(f"ΔΔScore rows: {len(df_delta)}")
    print(f"  Expected-affected: {df_delta['expected_affected'].sum()}")
    print(f"  Control interfaces: {(~df_delta['expected_affected']).sum()}")
    print("Saved → scores_delta.csv")
 
# ══════════════════════════════════════════════════════════════════
# STEP 9 — Benchmarking per Interface
# ══════════════════════════════════════════════════════════════════
 
print("\n" + "=" * 65)
print("STEP 9 — Benchmarking (ROC-AUC, PR-AUC, F1, Precision@Opt)")
print("=" * 65)
 
y_aligned    = df_aligned["binding"].astype(int)
all_results  = {}   # iface -> DataFrame
eval_ifaces  = interfaces + ["combined"]
 
for iface in eval_ifaces:
    if iface == "combined":
        sub   = df_aligned
        y_sub = y_aligned
        tag   = "combined"
    else:
        mask  = df_aligned["interface"] == iface
        sub   = df_aligned[mask]
        y_sub = y_aligned[mask]
        tag   = iface.replace(",", "")
 
    if y_sub.nunique() < 2:
        print(f"[{iface}] <2 classes — skipping.")
        continue
 
    rows = []
    for col in score_cols:
        s    = pd.to_numeric(sub[col], errors="coerce")
        mask = s.notna() & y_sub.notna()
        if mask.sum() < MIN_SAMPLES or y_sub[mask].nunique() < 2:
            continue
        y = y_sub[mask].astype(int).values
        s = s[mask].values
        try:
            roc = roc_auc_score(y, s)
            pr  = average_precision_score(y, s)
        except Exception:
            continue
        pre, rec, thr = precision_recall_curve(y, s)
        f1s      = (2 * pre[:-1] * rec[:-1]
                    / (pre[:-1] + rec[:-1] + 1e-10))
        best_idx = int(np.argmax(f1s))
        best_thr = float(thr[best_idx])
        y_pred   = (s >= best_thr).astype(int)
        best_f1  = f1_score(y, y_pred)
        best_pre = precision_score(y, y_pred, zero_division=0)
        top10    = y[np.argsort(s)[-10:][::-1]].mean()
        orig_dir = DIRECTIONS[col]
        rows.append({
            "Metric":           col,
            "Direction":        orig_dir,
            "ROC_AUC":          round(roc,      4),
            "PR_AUC":           round(pr,       4),
            "F1 (Opt)":         round(best_f1,  4),
            "Precision (Opt)":  round(best_pre, 4),
            "Top10_Precision":  round(top10,    4),
            "Opt_Threshold":    round(best_thr * orig_dir, 4),
            "n":                int(mask.sum()),
        })
 
    if not rows:
        print(f"[{iface}] No valid metrics."); continue
 
    res = pd.DataFrame(rows).sort_values("PR_AUC", ascending=False)
    res.to_csv(OUTPUT_DIR / f"rankings_{tag}.csv", index=False)
    all_results[iface] = res
    print(f"\n[{iface}] Top 10 by PR-AUC:")
    print(res[["Metric", "ROC_AUC", "PR_AUC",
               "F1 (Opt)", "Precision (Opt)"]].head(10).to_string(index=False))
    print(f"Saved → rankings_{tag}.csv")
 
# Main rankings.csv = combined interface results
if "combined" in all_results:
    all_results["combined"].to_csv(OUTPUT_DIR / "rankings.csv", index=False)
    print("\nMain rankings.csv saved (combined interfaces)")
 
# ══════════════════════════════════════════════════════════════════
# STEP 10 — Visualisations
# ══════════════════════════════════════════════════════════════════
 
print("\n" + "=" * 65)
print("STEP 10 — Visualisations")
print("=" * 65)
 
# ── Helper: quadrant plot ─────────────────────────────────────────
 
def quadrant_plot(ax, plot_df, m1, m2, t1, t2, d1, d2, title=""):
    """
    Scatter of m1 vs m2 (original scale) with TP/FP/TN/FN colouring.
    Labels only errors (FP, FN) with the sample name.
    t1, t2 are in original scale; d1, d2 are directions.
    """
    sub = plot_df.dropna(subset=[m1, m2]).copy()
    if sub.empty:
        return
 
    def classify(row):
        pred = ((row[m1] * d1) >= (t1 * d1) and
                (row[m2] * d2) >= (t2 * d2))
        actual = bool(row["binding"])
        if actual and pred:      return "TP"
        if not actual and pred:  return "FP"
        if actual and not pred:  return "FN"
        return "TN"
 
    sub["_cls"] = sub.apply(classify, axis=1)
 
    colors  = {"TP": "#2196F3", "TN": "#90CAF9",
               "FP": "#FF5722", "FN": "#FFCCBC"}
    markers = {"TP": "o", "TN": "o", "FP": "X", "FN": "X"}
 
    for cls, grp in sub.groupby("_cls"):
        ax.scatter(grp[m1], grp[m2],
                   c=colors[cls], marker=markers[cls],
                   s=90, alpha=0.85, label=cls,
                   edgecolors="white", linewidths=0.5)
 
    ax.axvline(t1, color="black", linestyle="--", alpha=0.35, lw=1)
    ax.axhline(t2, color="black", linestyle="--", alpha=0.35, lw=1)
 
    # Label errors
    for _, row in sub[sub["_cls"].isin(["FP", "FN"])].iterrows():
        ax.annotate(str(row["sample"])[-18:],
                    (row[m1], row[m2]),
                    fontsize=7, alpha=0.85,
                    xytext=(4, 4), textcoords="offset points")
 
    ax.set_xlabel(m1, fontsize=9)
    ax.set_ylabel(m2, fontsize=9)
    ax.set_title(title, fontsize=10, fontweight="bold")
    ax.legend(fontsize=8, markerscale=0.9)
 
 
# Use combined results for quadrant selection; fall back to first interface
quad_res  = all_results.get("combined",
             all_results[list(all_results.keys())[0]])
# Quadrant plots use original-scale df (not df_aligned) so thresholds
# are interpretable in native units
quad_src  = df   # original scale
 
for model, cols in model_groups.items():
    valid = [c for c in cols if c in quad_res["Metric"].values]
    if not valid:
        continue
 
    # ── 10a: Strip + box separability ───────────────────────────────
    valid_strip = [c for c in cols if df_aligned[c].notna().any()]
    if valid_strip:
        nc   = 4
        nr   = (len(valid_strip) + nc - 1) // nc
        fig, axes = plt.subplots(nr, nc, figsize=(20, 5 * nr))
        axes = np.atleast_1d(axes).flatten()
        for i, col in enumerate(valid_strip):
            ax  = axes[i]
            sub = df_aligned.dropna(subset=[col]).copy()
            sub["_lbl"] = sub["binding"].map(
                {True: "binder", False: "non-binder"})
            if sub.empty:
                continue
            sns.stripplot(data=sub, x="_lbl", y=col,
                          hue="_lbl", dodge=False, alpha=0.55,
                          jitter=0.25, palette=STRIP_PALETTE,
                          ax=ax, legend=False, size=5)
            sns.boxplot(data=sub, x="_lbl", y=col,
                        showcaps=False, fill=False, color="black",
                        width=0.4, ax=ax, zorder=10, showfliers=False)
            ax.set_title(col, fontsize=10, fontweight="bold")
            ax.set_xlabel("")
            ax.set_ylabel("Score (aligned)")
            ax.tick_params(labelsize=8)
        for j in range(i + 1, len(axes)):
            fig.delaxes(axes[j])
        fig.suptitle(f"{model.upper()} — Separability",
                     fontsize=16, y=1.01)
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / f"05_{model}_separability.png",
                    dpi=150, bbox_inches="tight")
        plt.close()
        print(f"Saved → 05_{model}_separability.png")
 
    # ── 10b: Quadrant plot (top-2 metrics for this model) ───────────
    model_rank = quad_res[quad_res["Metric"].isin(valid)].head(2)
    if len(model_rank) < 2:
        continue
    r1 = model_rank.iloc[0]
    r2 = model_rank.iloc[1]
    m1, t1, d1 = r1["Metric"], r1["Opt_Threshold"], r1["Direction"]
    m2, t2, d2 = r2["Metric"], r2["Opt_Threshold"], r2["Direction"]
 
    fig, ax = plt.subplots(figsize=(9, 8))
    quadrant_plot(ax, quad_src, m1, m2, t1, t2, d1, d2,
                  title=f"{model.upper()}: {m1} vs {m2}")
    plt.tight_layout()
    fname = (OUTPUT_DIR /
             f"06_{model}_quadrant_{m1.replace('/','_')}_vs_{m2.replace('/','_')}.png")
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Saved → {fname.name}")
 
# ── 10c: Rankings bar chart per interface ────────────────────────
 
for iface, res in all_results.items():
    top  = res.head(20)[::-1]
    tag  = iface.replace(",", "")
    fig, axes = plt.subplots(1, 2,
                             figsize=(14, max(5, len(top) * 0.42)),
                             sharey=True)
    for ax, col, xlabel, color in [
        (axes[0], "PR_AUC",  "PR-AUC",  "#1976D2"),
        (axes[1], "ROC_AUC", "ROC-AUC", "#388E3C"),
    ]:
        bars = ax.barh(range(len(top)), top[col], color=color, alpha=0.82)
        ax.axvline(0.5, color="gray", linestyle="--", lw=0.9,
                   label="random (0.5)")
        ax.set_xlim(0, 1.05)
        ax.set_xlabel(xlabel, fontsize=10)
        ax.set_title(xlabel, fontsize=11)
        ax.legend(fontsize=8)
        for bar, val in zip(bars, top[col]):
            ax.text(val + 0.01, bar.get_y() + bar.get_height() / 2,
                    f"{val:.3f}", va="center", fontsize=7)
    axes[0].set_yticks(range(len(top)))
    axes[0].set_yticklabels(top["Metric"].tolist(), fontsize=8)
    fig.suptitle(f"Top Metrics — [{iface}]", fontsize=13)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"07_rankings_{tag}.png",
                dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Saved → 07_rankings_{tag}.png")
 
# ── 10d: PR curves (top-5 per interface) ─────────────────────────
 
for iface, res in all_results.items():
    top5 = res.head(5)
    tag  = iface.replace(",", "")
    if iface == "combined":
        sub   = df_aligned
        y_sub = y_aligned
    else:
        mask  = df_aligned["interface"] == iface
        sub   = df_aligned[mask]
        y_sub = y_aligned[mask]
 
    fig, axes = plt.subplots(1, len(top5),
                             figsize=(4 * len(top5), 4), sharey=True)
    axes = np.atleast_1d(axes)
    for ax, (_, row) in zip(axes, top5.iterrows()):
        col  = row["Metric"]
        s    = pd.to_numeric(sub[col], errors="coerce")
        mask = s.notna() & y_sub.notna()
        if mask.sum() < 2:
            continue
        pre, rec, _ = precision_recall_curve(
            y_sub[mask].astype(int), s[mask])
        pr_auc = auc_fn(rec, pre)
        ax.plot(rec, pre, lw=2, color="#1976D2")
        ax.fill_between(rec, pre, alpha=0.1, color="#1976D2")
        ax.set_xlabel("Recall", fontsize=9)
        ax.set_ylabel("Precision", fontsize=9)
        ax.set_title(f"{col}\nPR-AUC={pr_auc:.3f}", fontsize=8)
        ax.set_xlim(0, 1); ax.set_ylim(0, 1)
        ax.grid(True, alpha=0.3)
    fig.suptitle(f"PR Curves — Top 5 [{iface}]", fontsize=12, y=1.01)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"08_pr_curves_{tag}.png",
                dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Saved → 08_pr_curves_{tag}.png")
 
# ── 10e: Confusion matrix (best metric, combined) ─────────────────
 
if "combined" in all_results:
    best_row   = all_results["combined"].iloc[0]
    best_col   = best_row["Metric"]
    opt_thresh = best_row["Opt_Threshold"]
    direction  = best_row["Direction"]
    sub_cm     = df.dropna(subset=[best_col]).copy()
    y_pred_cm  = ((sub_cm[best_col] * direction) >=
                  (opt_thresh * direction)).astype(int)
    cm = confusion_matrix(sub_cm["binding"].astype(int), y_pred_cm)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["Pred: Non-binder", "Pred: Binder"],
                yticklabels=["True: Non-binder", "True: Binder"])
    ax.set_title(f"Confusion Matrix: {best_col}\n"
                 f"(Threshold: {opt_thresh:.4f})", fontsize=11)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "09_confusion_matrix.png",
                dpi=150, bbox_inches="tight")
    plt.close()
    print("Saved → 09_confusion_matrix.png")
 
# ── 10f: ΔΔScore heatmaps ────────────────────────────────────────
 
if df_delta is not None and len(df_delta) > 0:
    delta_cols = [c for c in key_metrics if c in df_delta.columns]
    if delta_cols:
        for affected, label in [(True, "affected"), (False, "control")]:
            sub = df_delta[df_delta["expected_affected"] == affected]
            if sub.empty:
                continue
            pivot = sub.set_index("mutation")[delta_cols]
            fig, ax = plt.subplots(
                figsize=(max(8, len(delta_cols) * 0.6),
                         max(4, len(pivot) * 0.35)))
            sns.heatmap(pivot, ax=ax, cmap="RdBu_r", center=0,
                        xticklabels=pivot.columns,
                        yticklabels=pivot.index)
            ax.set_title(
                f"ΔΔScore — {'mutated' if affected else 'control'} interface"
                f"  (mutant − WT, direction-aligned)", fontsize=11)
            plt.xticks(rotation=45, ha="right", fontsize=8)
            plt.yticks(fontsize=8)
            plt.tight_layout()
            plt.savefig(OUTPUT_DIR / f"10_delta_heatmap_{label}.png",
                        dpi=150, bbox_inches="tight")
            plt.close()
            print(f"Saved → 10_delta_heatmap_{label}.png")
 
# ══════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("DONE — all outputs in:", OUTPUT_DIR)
print("=" * 65)
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {f.name}")

STEP 1 — Load & Filter Interfaces
Raw shape: (105, 70)
After interface filter {'A,C', 'B,C'}: (70, 70)
Rows per interface:
interface
A,C    35
B,C    35

STEP 2 — Parse Labels & Base Extraction
group
ala_scan    64
binder       6

Unique base structures: 3
Unique mutations:       23

Parsing preview (first 8 rows):
              sample interface    group         base mut_chain mutation
5ig7_abc_cut_B_G114A       A,C ala_scan 5ig7_abc_cut         B  B_G114A
5ig7_abc_cut_B_G114A       B,C ala_scan 5ig7_abc_cut         B  B_G114A
5ig7_abc_cut_A_K110A       A,C ala_scan 5ig7_abc_cut         A  A_K110A
5ig7_abc_cut_A_K110A       B,C ala_scan 5ig7_abc_cut         A  A_K110A
5ijk_acx_cut_A_I112A       A,C ala_scan 5ijk_acx_cut         A  A_I112A
5ijk_acx_cut_A_I112A       B,C ala_scan 5ijk_acx_cut         A  A_I112A
5ijk_acx_cut_B_Y107A       A,C ala_scan 5ijk_acx_cut         B  B_Y107A
5ijk_acx_cut_B_Y107A       B,C ala_scan 5ijk_acx_cut         B  B_Y107A

STEP 3 — Parse Array-Valued Column